# 05 — FastAPI + SSE 流式部署

> **目标**：将 Agent 部署为 HTTP 服务，支持流式输出  
> **产出**：`/chat/stream` SSE 端点 + `/chat` 同步端点

## 0. 启动服务

在终端执行（不要在 notebook 里跑）：

```bash
cd d:/Github项目/agent项目
uvicorn src.main:app --reload
```

然后访问 http://localhost:8000/docs 查看自动生成的 Swagger 文档。

## 1. 非流式测试（curl）

```bash
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "现在几点？"}'
```

## 2. 流式测试（curl）

```bash
curl -X POST http://localhost:8000/chat/stream \
  -H "Content-Type: application/json" \
  -d '{"message": "现在几点？", "thread_id": "test-1"}' \
  --no-buffer
```

输出示例：
```
data: ["updates", {"agent": {"messages": [{"content": "", "tool_calls": [{"name": "get_current_time", "args": {}}]}]}}]
data: ["updates", {"tools": {"messages": [{"content": "2026年07月03日 14:30:00", "name": "get_current_time"}]}}]
data: ["updates", {"agent": {"messages": [{"content": "现在是2026年7月3日下午2点30分。"}]}}]
data: [DONE]
```

## 3. 多轮对话测试

同一个 thread_id，第二条消息会记住第一条的上下文：

```bash
# 第一轮
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "我叫小明", "thread_id": "memory-test"}'

# 第二轮 —— Agent 应该记得
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "我叫什么名字？", "thread_id": "memory-test"}'
```

## 4. 用 Python 消费 SSE 流

如果需要在前端以外的场景消费流（比如另一个微服务），可以用 `httpx` 或 `aiohttp`。

In [ ]:
import httpx
import json

async def consume_sse():
    async with httpx.AsyncClient() as client:
        async with client.stream(
            "POST",
            "http://localhost:8000/chat/stream",
            json={"message": "1+1=?", "thread_id": "py-client"},
        ) as response:
            async for line in response.aiter_lines():
                if line.startswith("data: "):
                    data_str = line[6:]  # 去掉 "data: " 前缀
                    if data_str == "[DONE]":
                        print("流结束")
                        break
                    event = json.loads(data_str)
                    # event 是 (mode, payload) 元组
                    mode, payload = event
                    if mode == "updates":
                        for node, update in payload.items():
                            if "messages" in update:
                                msg = update["messages"][-1]
                                role = msg.get("type", "?")
                                content = str(msg.get("content", ""))[:80]
                                print(f"[{node}] {role}: {content}")
                    elif mode == "messages":
                        # 逐 token 流（打字机效果）
                        content = str(payload[0].get("content", ""))
                        print(content, end="", flush=True)

# 在设置有 event loop 的环境下运行
# import asyncio
# asyncio.run(consume_sse())

## 5. 架构总结

```
┌──────────┐   HTTP POST    ┌──────────┐   invoke/astream   ┌──────────┐
│  浏览器   │ ───────────────→ │ FastAPI   │ ─────────────────→ │ LangGraph │
│ (前端)   │ ←─── SSE ───── │ (src/main)│ ←─── events ────── │ (agent)  │
└──────────┘                └──────────┘                    └──────────┘
```

**SSE vs WebSocket**：Agent 的交互是"请求→流式响应"模式，单向推送就够。
SSE 更简单、兼容性更好，是当前 Agent 部署的主流选择。

**下一步**：在阶段 F 中，把 MemorySaver 替换为 SqliteSaver，实现持久化。